# **02 - Exploratory Data Analysis (EDA)**
---

## **Table of Content**
[Objective](#obj)

[Dataset Info](#data-info)

[Domain-Related Questions](#question)

[Summary](#summary)

[Import Necessary Libraries](#import)

[Import Raw Dataset and Preprocess Using Method](#preprocess)

[Initial Inspection](#inspect)

[Question-Driven Analysis](#analysis)
* [Question 1](#q1)
* [Question 2](#q2)
* [Question 3](#q3)
* [Question 4](#q4)



---

## <a id ='obj'></a>**Objective**
To uncover temporal patterns, seasonalities and feature correlations to inform effective feature engineering and data preprocessing for accurate time series forecasting.

## <a id ='data-info'></a> **Dataset Info**
* **Source:** UCI Machine Learning Repository
* **Title:** Appliances Energy Prediction
* **Time Range:** 11th January 2016 - 27th May 2016
* **Primary Target Variable:** `Appliances` (Watt-hour consumed)

## <a id = 'question'></a> **Domain-Related Questions**
In this section, we'll define the set of questions we'd like to answer from this EDA.

> **Q1**: What are the dominant daily and weekly consumption cycles, and does the habits stay the same from January to May?

> **Q2**: Does downsampling from 10-minute intervals to 30-minute or 1-hour steps preserve key signals while reducing noise?

> **Q3**: How strongly does the current electrical consumption depends on past information, and how far back does the influence persist?

> **Q4**: Which environmental variables correlate most strongly with appliance energy usage, and are these relationships immediate or lagged?

## <a id = 'summary'></a>**Summary**

### Answer to Questions
> **Q1: What are the dominant daily and weekly consumption cycles, and does the habits stay the same from January to May?**
>
> *Daily Consumption Pattern*
> 
> Follows a consistent diurnal pattern across all months: lowest overnight, steady rise through the day, peaking in the evening.
> 
> *Weekly Consumption Pattern*
> 
> Surprising peak on Monday, low baseline Tue–Thu, heavy activity Fri–Sat, and a sharp drop on Sunday. 
> 
> *Consistency from January to May*
> 
> Daily pattern (plotted by hour) remain consistent for 5 months. Unlike the daily pattern, peak days shift month-to-month.


> **Q2: Does downsampling from 10-minute intervals to lower frequency steps preserve key signals while reducing noise?**
> 
> *Effects of Downsampling*
> 
> As we decrease the sampling frequency, the plot becomes smoother, but it deviates from the raw plots as we decrease frequency beyond 1 sample per 30 minutes.
> 
> *A good trade-off*
> 
> 1 sample/30-min shows the best trade-off between removing noise and preserving key signals.

> **Q3: How strongly does the current electrical consumption depends on past information, and how far back does the influence persist?**
>
> *Past Information Influence on Appliances*
> 
> Consumption from recent past influence future readings strongly, with the highest correlation coefficient of ~0.75. However, influence decays over time.
>
> *Influence Persistence*
>
> Independent predictive power of lagged feature remains significant up until the 120-minutes mark.
>
> *Seasonality Pattern*
>
> Although influence decays over time, the influence of consumption from exactly 1, 5 and 7 days ago exhibits a local maxima by a great margin.

> **Q4: Which environmental variables correlate most strongly with appliance energy usage, and are these relationships immediate or lagged?**
>
>
> *Correlation Considering Past Information*
> * Light is a strong predictive feature, but only in short-term. Moving past the 1-hour mark, coefficient value drop by ~36%
>
> * In contrast, temperature and humidity for both living room and outdoor have a more stable long-term influence, albeit light outperform it in a more recent past.
>
> * Bathroom humidity have a delayed influence, where influence gradually increases to it's peak at 80 - 100 minutes mark.
>
> *Immediate vs Lagged Relationship Classification*
>
> * **Immediate relationship**: lights, living room & outdoor temperature/humidity
>
> * **Lagged relationship**: bathroom humidity



### Decisions
 1. *Keep 'hour' as a predictive feature for the model, but discard the 'day'* 

    **Justification**: As stated, daily pattern (plotted by hour) remain consistent throughout the 5 month period, but peak days changed from month-to-month. 

2. *Downsample to 1 sample/30 min*

    **Justification**: Best trade-off between removing noise and preserving key signals.

3. *Create target feature lagged variable for 10-120 minutes, as well as 1, 5 and 7 days*

    **Justification**: Strong independent predictive power

4. *Keep light lagged variables short-term (10 - 60 minutes back)*

    **Justification**: Light have decaying predictive power past the 1-hour mark.

5. *No lag features created for living room and outdoor temperature and humidity*

    **Justification**: As their influence remains consistent for long period of time, adding lag variable will add redundancy.

6. *Keep Only 80-120 minutes Lagging for Bathroom Humidity*

    **Justification**: Captures the physical time delay of this feature influence.


---
## <a id = 'import'></a> **Import Necessary Libraries**

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import math
from statsmodels.graphics.tsaplots import plot_pacf
import numpy as np
from statsmodels.tsa.stattools import ccf


# Add the root directory (one level up from notebooks/) to Python's path
root_dir = Path.cwd().parent
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

from src.preprocess import preprocess_energy

## <a id = 'preprocess'></a> **Import Raw Dataset**

In [ ]:
PROJECT_ROOT = Path.cwd().parent

# Import csv file
energy = pd.read_csv(f'{PROJECT_ROOT}/data/raw/energydata_complete.csv')

In [ ]:
# Print samples
energy.head(3)

## **Preprocess Using Method**
We'll use the reusable method we developed in preprocessing stage. Refer *notebooks/01_data_preprocessing.ipynb* for more details.

In [ ]:
# Call preprocess_energy method
energy = preprocess_energy(energy)

# Print preprocessed samples
energy.head(3)

## <a id = 'inspect'></a>**Initial Inspection**

We'll first inspect the dataset characteristics, such as the shape, data types and so on.

In [ ]:
# Inspect shape
print(f'Number of Rows: {energy.shape[0]}')
print(f'Number of Columns: {energy.shape[1]}')

# Inspect statistics
print('\n\n---DATASET INFORMATION---')
print(energy.info())

In [ ]:
# Print statistics overview
energy.describe()

## <a id = 'analysis'></a> **Question-Driven Analysis**

Every question in this section follows a standardized 4-step structure:

1. **Question**: The target question being investigated.

2. **Hypothesis**: Anticipated finding based on domain context or prior logic.

3. **Analysis**: Iterative plots paired directly with analytical notes and observations.

4. **Decision**: Decisions taken based on the observation and any conclusion drawn.

<br>

---

> ### <a id = 'q1'></a> **Q1**: *What are the dominant daily and weekly consumption cycles, and does the habits stay the same from January to May?*



#### **Hypothesis**
Daily consumption will **peak during daytime/evening hours** and **drop overnight** during sleep, with **higher overall volume on weekends** due to home occupancy. Furthermore, as January - May is not a long enough period for major seasonal change, the consumption pattern of each month should be relatively similar to one another.

#### **Analysis**

#### A Typical Daily Consumption Pattern
We'll aggregate the whole 4-months data by hour, calculate it's mean and plot it into a time-series plot.

In [ ]:
# Create canvas
fig, ax = plt.subplots(figsize=(12, 5))

# Plot the aggregated trend
sns.lineplot(
    data=energy, 
    x='hour', 
    y='Appliances', 
    color="orange", 
    marker='o', 
    linewidth=2,
    ax=ax,
    label = 'Appliances Energy',
    errorbar=('ci', False)
)

# Display settings
ax.set_xticks(range(0, 24))
ax.set_xticklabels([f"{h}:00" for h in range(24)], rotation=45)

plt.title("Appliance Energy Throughout Time of Day", fontsize=20)
plt.xlabel("Time of Day", fontsize=14)
plt.ylabel("Average Consumption", fontsize=14)
plt.grid(True, linestyle="--", alpha=0.5)

# Show the graph
plt.tight_layout()
plt.show()

> #### Observations
> **```1. Lowest baseline overnight```**
> 
> From 12am to 6am, the energy consumption is at it's lowest baseline, due to residence inactivity. Those consumption probably comes from the always-on appliances such as the fridge.
>
> **```2. Residents are active in morning, and take breaks in the afternoon```**
>
> Starting 6 am, consumption steadily rises to a local maximum at 11 am. But then, it decreases steadily until 3 pm.
>
> **```3. Consumption peaks in the evening (19:00 hrs)```**
>
> Around the time people coming back from work (4 pm - 7 pm), consumption shows an absolute highest reading, where residents are probably showering, eating dinner or settling down into the house.
>
> **```4. Steady decline through the midnight into low baseline again```**
>
> To loop through the cycle, the consumption decline again into lowest baseline, suggesting residents' transition into inactivity.

#### A Typical Weekly Consumption Pattern
We'll do the same thing again, but this time grouping by day of the week.

In [ ]:
# Create canvas
fig, ax = plt.subplots(figsize=(12, 5))

# Plot the aggregated trend
sns.lineplot(
    data=energy, 
    x='day', 
    y='Appliances', 
    color="orange", 
    marker='o', 
    linewidth=2,
    ax=ax,
    label = 'Appliances Energy',
    errorbar=('ci', False)
)

# Display settings
plt.title("Appliance Energy Throughout Day of the Week", fontsize=20)
plt.xlabel("Day", fontsize=14)
plt.ylabel("Average Consumption", fontsize=14)
plt.grid(True, linestyle="--", alpha=0.5)

# Show the graph
plt.tight_layout()
plt.show()

>#### Observations
> **```1. Highest consumption on Monday```**
>
> Contradicting our hypothesis, consumption typically peaks on Mondays, instead of the weekend. But we're not sure if that's the case for any timeframe (whether pattern changes in different month etc.), so we'll check on this later.
>
> **```2. Plateau from Tuesday - Thursday```**
>
> This supports our initial hypothesis that consumption will stay relatively low in the weekday, where most people spend their time at work/school.
>
> **```3. Higher Friday/Saturday consumption```**
>
> **```4. Lower consumption on Sunday```**

#### Change in Weekly Consumption Pattern from January to May
To see if the 'typical' pattern holds true from January to May, we'll group them by month instead.

In [ ]:
# Create canvas
fig, ax = plt.subplots(nrows=5, ncols=1, figsize=(8, 12), sharex=True)

months = ['January', 'February', 'March', 'April', 'May']

# Loop through each day and plot it on its respective subplot row
for i, month in enumerate(months):
    df = energy[energy['month'] == month]
    
    sns.lineplot(
        data=df, 
        x='hour', 
        y='Appliances', 
        linewidth=1.5,
        ax=ax[i],
        label=month,
        marker = 'o',
        legend = False,
        errorbar=('ci', False)
    )
    
    ax[i].set_ylabel("Average Consumption")
    ax[i].set_xlabel("Hour")
    ax[i].grid(True, linestyle="--", alpha=0.4)
    ax[i].set_title(f'{month} Daily Usage Trend', fontsize = 14)
    ax[i].set_xticks([h for h in range (24)])
    ax[i].set_xticklabels([f'{h}:00' for h in range (24)])

# Axis settings
for axs in ax.flat:
    axs.tick_params(labelbottom=True)
    axs.tick_params(axis='x', labelrotation=45)

# Display graph
plt.tight_layout()
plt.show()

#### Change in Daily Consumption Pattern from January to May

In [ ]:
# Create canvas
fig, ax = plt.subplots(nrows=5, ncols=1, figsize=(6, 12), sharex=True)

# Define month for label
months = ['January', 'February', 'March', 'April', 'May']

# Loop through each day and plot it on its respective subplot row
for i, month in enumerate(months):
    df = energy[energy['month'] == month]
    
    sns.lineplot(
        data=df, 
        x='day', 
        y='Appliances', 
        linewidth=1.5,
        ax=ax[i],
        marker= 'o',
        label=month,
        legend = False,
        errorbar=('ci', False)
    )
    
    ax[i].set_ylabel("Average Consumption")
    ax[i].grid(True, linestyle="--", alpha=0.4)
    ax[i].set_title(f'{month} Usage Trend', fontsize = 14)
    

for axs in ax.flat:
    axs.tick_params(labelbottom=True)
    axs.tick_params(axis='x', labelrotation=45)

plt.tight_layout()
plt.show()

> #### Observations
>
> **```1. Daily consumption pattern stays visibly the same, with minor variations```**
>
> **```2. Weekly consumption pattern changes drastically```**
>

#### **Decisions**

1. *Keep 'hour' as a predictive feature for the model, but discard the 'day'*

    **Justification**: The daily pattern, grouped by 'hour' remains visibly similar throughout the entire dataset period. This means that 'hour' is a reliable predictive feature. The vice versa applies for 'day'.


---
> ### <a id = 'q2'></a> **Q2**: *Does downsampling from 10-minute intervals to lower frequency steps preserve key signals while reducing noise?*

#### **Hypothesis**
Downsampling to lower frequency steps  will **smooth out sensor noise**, but it may fail to preserving the core peaks and fluctuations required for modelling when the downsampling is done too aggressively. A good trade-off should exist somewhere between 20 minutes - 1 hour step size.

#### **Analysis**

#### Choosing A Sample Week

Because we're dealing with a massive amount of rows (over 19k rows), plotting all of them to see the effect of downsampling would be overwhelmingly cluttered. For that, we'll only use the **median 7 days** data.

In [ ]:
# Slice median week
mid = math.ceil(len(energy)/2)
med_week = energy.iloc[mid:mid + 1008]

#### Experiment: Downsampling Effect

We'll consider these downsampling step size:
* 20 minutes
* 30 minutes
* 60 minutes

Downsampling will also be done using max aggregation method.

**Justification**: Peak energy usage usually matter more compared to average usage, as we do not want electricity cut-off when usage exceed the normal amount.

In [ ]:
steps = ['20min', '30min', '1h']

fig, ax = plt.subplots(nrows=3, ncols=1, figsize=(12, 10), sharex=True)

for i, item in enumerate(steps):
    # 1. Plot raw data
    sns.lineplot(
        data=med_week,
        x=med_week.index, 
        y='Appliances', 
        label='Raw Appliances',
        ax=ax[i], 
        alpha=0.3
    )

    # 2. Resample 
    downsampled = med_week['Appliances'].resample(item).max()

    # 3. Plot downsampled reading
    sns.lineplot(
        x=downsampled.index,
        y=downsampled.values,
        ax=ax[i],
        label=f'Downsampled Appliances: 1 sample/{item}',
        color='red'
    )

    # Display settings
    ax[i].set_title(f"10-min vs {item} Steps Appliances Reading", fontsize=14)
    ax[i].set_ylabel("Appliances")
    ax[i].grid(True, linestyle="--", alpha=0.5)

ax[-1].set_xlabel("Date")
plt.tight_layout()
plt.show()

>#### Observations
>
> **```1. Loss of information in the 1h step size```**
>
> **```2. Insignificant smoothing in 20 min step size```**
>
> **```3. 30 min step size is a balanced version```**

#### **Decisions**

1. *Downsample to 1 sample/30 min*

    **Justification**: The original 1 sample/10-min frequency have too much noise. The 1 sample/30-min downsampling proves to capture major peaks shown by the raw dataset, but significantly reduces minor fluctuations that will disrupt model performance.

---
> ### <a id = 'q3'></a> **Q3**: *How strongly does the current electrical consumption depends on past information, and how far back does the influence persist?*

#### **Hypothesis**
Electrical consumption highly correlates with the recent past consumption. However, as we see a consistent daily pattern from the previous analysis, a more distant past such as 24-hour lag can also be influential to the current consumption, depending on seasonal cycle.

#### **Analysis**

#### Partial Autocorrelation Function Analysis (PACF) for Appliances
For this experiment, we'll consider lagged feature in 4 different aspects:
* Short-term dependence: 0 - 3 hours lag
* Long-term dependence: 0 - 7 days lag

#### Short-term Dependence

In [ ]:
# Create canvas
fig, ax = plt.subplots(figsize=(8, 5))

# Plot the PACF
plot_pacf(
    energy['Appliances'],
    lags=3*6, 
    ax = ax)


plt.title("Short-Term PACF")
plt.xlabel("Lags (Minutes)")
plt.ylabel("Correlation Coefficient")


ax.set_xticks(np.arange(0, 19, 3))
ax.set_xticklabels(i*30 for i in range (7))
plt.grid()
plt.show()

>#### Observations
>
> ```1. Lag 10 minutes captures almost all of the independent predictive memory```
>   
> Indicated by sharp coefficient drop after the 10 minutes lag, with absolute difference of  ~0.7 (between 10 and 20 minutes lag)
>
> ```2. Coefficient drops below the significance threshold after the 120 minutes mark```
>
> Beyond the 120-minutes mark, the dependence is considered statistically insignificant (indistinguishable from random noise / zero).

#### Long-Term Dependence
Because 1008 samples are visibly hard to see in the plot, we'll resample the data first into 30-min step size (per our findings in Q2 analysis that 30-min step size conserve the major fluctuations).

In [ ]:
# Resample to 30-min step size
energy_resample = energy["Appliances"].resample("30min").max()

# Create canvas
fig, ax = plt.subplots(figsize=(12, 5))

# Plot the PACF
plot_pacf(
    energy_resample,
    lags=336, 
    ax = ax)


plt.title("Long-Term PACF")
plt.xlabel("Lags (Days)")
plt.ylabel("Correlation Coefficient")

ax.set_xticks(np.arange(0, 337, 48))
ax.set_xticklabels(i for i in range (8))

# Zoom into y-axis to make bumps more visible
ax.set_ylim([-0.15, 0.15])

plt.grid()
plt.show()

> #### Observations
>
> ```1. Relatively larger coefficient every 24 hours```
>
> This implicates strong daily seasonality cycle (consumption at 3 pm yesterday may be similar to 3 pm today)
>
> ```2. Highest coefficient occurs on 1, 5 and 7 lag days```
>
> This suggests a strong predictive power. Perhaps we should include these lag into consideration when building the model.

#### **Decisions**
1. *Create lagged variable for 10-120 minutes, as well as 1, 5 and 7 days*

    **Justification**: Strong independent predictive power, which includes short-term memory and seasonality pattern information.

---
> ### <a id = 'q4'></a> **Q4**: *Which environmental variables correlate most strongly with appliance energy usage, and are these relationships immediate or lagged?*

#### **Hypothesis**
Lighting and indoor/outdoor temperatures correlate most strongly with appliance energy usage, exhibiting immediate effects driven by human activity alongside delayed responses from thermal inertia.

#### **Analysis**
We'll do the analysis in two part:
* Correlation Analysis, taking into account only the present information for Appliances
* Cross Correlation Function (CCF) Analysis, taking into account lag variables

#### Correlation Analysis (Only Using Present Sensor Reading)

In [ ]:
corr_matrix = energy.drop(['hour', 'rv1', 'rv2'], axis =1).corr(numeric_only=True)

# Plot the heatmap
plt.figure(figsize=(16, 2))
sns.heatmap(corr_matrix.head(1).drop(columns='Appliances'), annot=True, cmap='coolwarm', vmin=-0.2, vmax=0.2, fmt=".2f")

# Display the plot
plt.title("Correlation Heatmap (Appliances Only)")
plt.xticks(rotation =0)
plt.show()

> #### Observations
>
> ```1. Highest influencing features```
>
> Lights, living room temperature and humidity, and outdoor temperature and humidity have relatively high correlations to electrical consumption.

#### CCF Analysis
Based on our findings, we're considering these lag numbers:
* 10 minutes - 120 minutes (1 - 12 step)
* 1 day (144 steps)
* 5 days (720 steps)
* 7 days (1008 steps)

In [ ]:
lag = [i+1 for i in range (12)] + [144, 720, 1008]

sensor_features = energy.select_dtypes(include = 'number').columns.drop(['Appliances', 'rv1', 'rv2', 'hour'])

ccf_val = pd.DataFrame()
ccf_filtered = pd.DataFrame()

for col in sensor_features:
    # Compute CCF
    ccf_val = ccf(energy["Appliances"], energy[col], adjusted=False, nlags=1009)
    # Filter ONLY the specific lags requested
    ccf_filtered[col] = ccf_val[lag]
    
ccf_filtered['lag'] = lag

ccf_filtered = ccf_filtered.set_index("lag")

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(
    ccf_filtered, annot=True, cmap="coolwarm", vmin=-0.2, vmax=0.2, fmt=".2f"
)

plt.title("Cross-Correlated Heatmap")
plt.xlabel("Sensors")
plt.ylabel("Lags")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

> #### Observations
>
> ```1. Lights is a strong short-term and seasonal feature ```
>
> Strongest overall influence in recent past, but decays rapidly as time pass. By the 1-hour mark, the influence reduce by ~36%.
>
> ```2. Outdoor and living room temperature and humidity is a stable long-term predictor ```
>
> Indicated by consistent color in the column for T_liv_room, RH_liv_room, T_outdoor and RH_outdoor, we can say that inertia for these variables are high. This shows us that there's no need for lag variables for these features, as the present value contains all the necessary information already.
>
> ```3. Bathroom humidity shows a delayed influence ```
>
> Bathroom humidity shows low influence in the recent past, but started to have significant predictive power by the 80-120 minutes lag mark.


#### **Decisions**
1. *Keep light lagged variables short-term (10 - 60 minutes back)*

    **Justification**: Light have decaying predictive power past the 1-hour mark.

2. *No lag features created for living room and outdoor temperature and humidity*

    **Justification**: As their influence remains consistent for long period of time, adding lag variable will add redundancy.

3. *Keep Only 80-120 minutes Lagging for Bathroom Humidity*

    **Justification**: Captures the physical time delay of this feature influence.